# Use Case 1 — Portfolio Heatmap & Simulation

Croises les dimensions véhicule (marque, modèle, motorisation, CO₂) avec les dimensions crédit/financières (ratings, secteur, exposition) en 3 métriques : Volume, Concentration Financière, Intensité Risk Asset.

**Sections :**
1. Chargement et préparation des données
2. Heatmap interactive (filtres Brand / Country)
3. Mode simulation (add/remove brands → 3 panneaux comparatifs)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from ipywidgets import interact, interactive, HBox, VBox, Output
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
DATA_DIR = Path('../data')

## 1. Chargement des données

In [ ]:
# Load all parquets — ~21k rows, fast
files = sorted(DATA_DIR.glob('NOVA - *.parquet'))
print(f'Loading {len(files)} files...')

COLS = [
    'ID_CONTRACT', 'VEHICLE_ID', 'ID_QUOTATION', 'COB_DATE',
    'COUNTRY', 'BRAND_UPDATE', 'POWER_CATEGORY', 'VA_CO2_EMSS_REAL',
    'NOVA_ASSET_STATUS', 'BIKE_OR_CAR',
    'OBLIGOR_IDENTIFIER', 'GROUP_RATING', 'COUNTERPARTY_RATING',
    'CLS_GROUP_RATING', 'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION',
    'SHARED_CLIENT_FLAG', 'VEHICLE_PRICE_EUR',
    'EXPOSURE_AMOUNT_LTR', 'EXPOSURE_AMOUNT_MTR',
]

nova = pd.concat(
    [pd.read_parquet(f, columns=COLS) for f in files],
    ignore_index=True
)

# Derived field
nova['EXPOSURE_AMOUNT_TOT'] = nova['EXPOSURE_AMOUNT_LTR'] + nova['EXPOSURE_AMOUNT_MTR']

# CO2 bucket for display
def co2_bucket(val):
    try:
        v = int(float(val))
        lo = (v // 10) * 10
        return f'[{lo}-{lo+9}]'
    except:
        return 'UNK'

nova['CO2_BUCKET'] = nova['VA_CO2_EMSS_REAL'].apply(co2_bucket)

# Keep latest snapshot per contract to avoid duplicates across months
nova = nova.sort_values('COB_DATE').drop_duplicates(
    subset=['ID_CONTRACT', 'VEHICLE_ID', 'ID_QUOTATION'], keep='last'
)

print(f'Loaded: {len(nova):,} rows | {nova["COUNTRY"].nunique()} countries | {nova["BRAND_UPDATE"].nunique()} brands')
nova[['GROUP_RATING','COUNTERPARTY_RATING','CLS_GROUP_RATING',
      'EXPOSURE_AMOUNT_LTR','SHARED_CLIENT_FLAG',
      'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION','VEHICLE_PRICE_EUR','OBLIGOR_IDENTIFIER']].head(5)

Loading 1056 files...
Loaded: 1,124,965 rows | 8 countries | 26 brands


,GROUP_RATING,COUNTERPARTY_RATING,CLS_GROUP_RATING,EXPOSURE_AMOUNT_LTR,SHARED_CLIENT_FLAG,ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION,VEHICLE_PRICE_EUR,OBLIGOR_IDENTIFIER
539610,03,02+,3.0,14921.25,SHARED CLIENT BY DEFAULT,CONSTRUCTION & REAL ESTATE,24086.90,LU43770226
539611,03,02+,3.0,0.00,SHARED CLIENT BY DEFAULT,CONSTRUCTION & REAL ESTATE,36346.30,LU43770226
382409,02,02+,5.0,0.00,NON-SHARED CLIENT,FINANCIAL SERVICES,20892.15,IT53052421
292721,02-,03-,5.0,0.00,SHARED CLIENT,TRANSPORT EQUIPMENT,24945.42,DE55870242
697214,04,02+,3.0,39883.01,SHARED CLIENT,PROFESSIONAL SERVICES,68971.53,ES24234835


## 2. Heatmap interactive — avec filtres Brand & Country

In [ ]:
RATING_COLS = ['CLS_GROUP_RATING', 'COUNTERPARTY_RATING', 'GROUP_RATING']

def format_number(val):
    if pd.isna(val):
        return ''
    if val >= 1_000_000:
        return f'{val/1_000_000:.1f}M'
    elif val >= 1_000:
        return f'{val/1_000:.1f}k'
    return f'{int(val)}'


def get_grouped_data(df, y_col, x_cols, metric='volume'):
    is_rating = any(col in x_cols for col in RATING_COLS)
    unique_keys = (['OBLIGOR_IDENTIFIER'] if is_rating else []) + [
        'ID_CONTRACT', 'VEHICLE_ID', 'ID_QUOTATION'
    ]
    # only keep keys that exist in df
    unique_keys = [k for k in unique_keys if k in df.columns]

    df_clean = df.dropna(subset=[y_col] + list(x_cols)).drop_duplicates(
        subset=unique_keys + [y_col] + list(x_cols)
    )
    if df_clean.empty:
        return pd.DataFrame()

    group_cols = [y_col] + list(x_cols)
    if metric == 'concentration_financiere':
        df_g = df_clean.groupby(group_cols)['EXPOSURE_AMOUNT_TOT'].sum().reset_index(name='count')
    elif metric == 'intensite_risk_asset':
        df_g = df_clean.groupby(group_cols)['VEHICLE_PRICE_EUR'].sum().reset_index(name='count')
    else:
        df_g = df_clean.groupby(group_cols).size().reset_index(name='count')

    df_g['x_combined'] = df_g[list(x_cols)].astype(str).agg(' | '.join, axis=1)
    pivot = df_g.pivot_table(
        index=y_col, columns='x_combined', values='count', aggfunc='sum', fill_value=0
    )
    # Sort columns by total descending
    pivot = pivot[pivot.sum().sort_values(ascending=False).index]
    return pivot


def plot_heatmap(df_pivot, title, metric='volume', page=0,
                 cmap='YlGnBu', ax=None):
    rows_per_page = 30
    total_rows = len(df_pivot)
    df_sub = df_pivot.iloc[page * rows_per_page: (page + 1) * rows_per_page]
    if df_sub.empty:
        print('No data for this page.')
        return

    annot = df_sub.map(format_number)

    if ax is None:
        height = max(6, len(df_sub) * 0.45)
        fig, ax = plt.subplots(figsize=(min(22, 2 + len(df_sub.columns)), height))

    sns.heatmap(
        df_sub, annot=annot, fmt='', cmap=cmap,
        annot_kws={'size': 9}, linewidths=0.3, linecolor='#e0e0e0',
        ax=ax
    )
    ax.set_title(f'{title}\n({total_rows} lignes total)', fontsize=11, pad=8)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)


def interactive_heatmap(df):
    y_options  = ['BRAND_UPDATE', 'POWER_CATEGORY', 'CO2_BUCKET', 'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION']
    x_options  = ['CLS_GROUP_RATING', 'COUNTERPARTY_RATING', 'GROUP_RATING',
                  'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION', 'SHARED_CLIENT_FLAG']
    metric_opts = [
        ('Volume (nb contrats)', 'volume'),
        ('Concentration Financière (EUR)', 'concentration_financiere'),
        ('Intensité Risk Asset (prix véhicules)', 'intensite_risk_asset'),
    ]

    all_brands    = sorted(df['BRAND_UPDATE'].dropna().unique())
    all_countries = sorted(df['COUNTRY'].dropna().unique())

    y_w       = widgets.Dropdown(options=y_options, value='BRAND_UPDATE',
                                  description='Axe Y:', style={'description_width': '70px'})
    x_w       = widgets.SelectMultiple(options=x_options, value=[x_options[0]],
                                        description='Axe X:', rows=5,
                                        style={'description_width': '70px'})
    m_w       = widgets.Dropdown(options=metric_opts, description='Métrique:',
                                  style={'description_width': '70px'})
    brand_w   = widgets.Dropdown(options=['All'] + all_brands, value='All',
                                  description='Brand:', style={'description_width': '70px'})
    country_w = widgets.Dropdown(options=['All'] + all_countries, value='All',
                                  description='Country:', style={'description_width': '70px'})
    page_w    = widgets.IntSlider(min=0, max=10, step=1, value=0,
                                   description='Page:', continuous_update=False,
                                   layout=widgets.Layout(display='none'))
    out       = Output()

    def refresh(*_):
        df_f = df.copy()
        if brand_w.value != 'All':
            df_f = df_f[df_f['BRAND_UPDATE'] == brand_w.value]
        if country_w.value != 'All':
            df_f = df_f[df_f['COUNTRY'] == country_w.value]

        pivot = get_grouped_data(df_f, y_w.value, list(x_w.value), metric=m_w.value)
        page_w.layout.display = 'flex' if len(pivot) > 30 else 'none'
        page_w.max = max(0, len(pivot) // 30)

        with out:
            out.clear_output(wait=True)
            if pivot.empty:
                print('No data for this selection.')
                return
            plot_heatmap(
                pivot,
                title=f'{y_w.value} vs {list(x_w.value)} — {m_w.value}',
                metric=m_w.value,
                page=page_w.value,
            )
            plt.tight_layout()
            plt.show()

    for w in [y_w, x_w, m_w, brand_w, country_w, page_w]:
        w.observe(refresh, names='value')

    filters = HBox([brand_w, country_w, m_w])
    axes    = HBox([y_w, x_w])
    display(VBox([filters, axes, page_w, out]))
    refresh()


interactive_heatmap(nova)

## 3. Mode Simulation — Add / Remove vehicles par Brand

Sélectionne une ou plusieurs marques, ajuste les quantités, et compare les 3 heatmaps : **Actuel · Simulé · Delta**.

In [4]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def get_brand_rating_distribution(df, brand):
    """Extract empirical rating probability distribution for a given brand."""
    subset = df[df['BRAND_UPDATE'] == brand]['GROUP_RATING'].dropna()
    if subset.empty:
        return None, None
    counts = subset.value_counts(normalize=True)
    return counts.index.tolist(), counts.values.tolist()


def sample_vehicles_for_brand(df, brand, n, rng):
    """
    Generate n synthetic rows for brand, sampling from existing brand rows
    and assigning ratings according to the brand's empirical distribution.
    """
    template = df[df['BRAND_UPDATE'] == brand]
    if template.empty or n == 0:
        return pd.DataFrame()

    sampled = template.sample(n=n, replace=True, random_state=rng.integers(1e6)).copy()

    # Assign new unique identifiers
    sampled['ID_CONTRACT']   = ['SIM_' + str(i) for i in range(n)]
    sampled['VEHICLE_ID']    = ['SIM_' + str(i) for i in range(n)]
    sampled['ID_QUOTATION']  = ['SIM_' + str(i) for i in range(n)]
    sampled['OBLIGOR_IDENTIFIER'] = ['SIM_OBL_' + str(i) for i in range(n)]

    # Re-assign ratings from empirical brand distribution
    ratings, probs = get_brand_rating_distribution(df, brand)
    if ratings:
        sampled['GROUP_RATING'] = rng.choice(ratings, size=n, p=probs)

    # Price slight jitter
    price_std = template['VEHICLE_PRICE_EUR'].std()
    sampled['VEHICLE_PRICE_EUR'] = np.clip(
        sampled['VEHICLE_PRICE_EUR'] + rng.normal(0, price_std * 0.1, n),
        1000, None
    ).round(2)
    sampled['EXPOSURE_AMOUNT_LTR'] = (sampled['VEHICLE_PRICE_EUR'] * rng.uniform(0.4, 0.65, n)).round(2)
    sampled['EXPOSURE_AMOUNT_MTR'] = (sampled['VEHICLE_PRICE_EUR'] * rng.uniform(0.05, 0.15, n)).round(2)
    sampled['EXPOSURE_AMOUNT_TOT'] = sampled['EXPOSURE_AMOUNT_LTR'] + sampled['EXPOSURE_AMOUNT_MTR']

    return sampled


def apply_simulation(df_orig, add_dict, remove_dict, rng):
    """
    Returns simulated df:
    - add_dict:    {brand: n_to_add}
    - remove_dict: {brand: n_to_remove}  (removes randomly from existing rows)
    """
    df_sim = df_orig.copy()

    # Remove
    for brand, n in remove_dict.items():
        if n <= 0:
            continue
        idx = df_sim[df_sim['BRAND_UPDATE'] == brand].index
        n = min(n, len(idx))
        drop_idx = rng.choice(idx, size=n, replace=False)
        df_sim = df_sim.drop(index=drop_idx)

    # Add
    new_rows = []
    for brand, n in add_dict.items():
        if n <= 0:
            continue
        new_rows.append(sample_vehicles_for_brand(df_orig, brand, n, rng))

    if new_rows:
        df_sim = pd.concat([df_sim] + new_rows, ignore_index=True)

    return df_sim


def plot_three_panels(pivot_orig, pivot_sim, y_col, x_cols, metric):
    """3-panel figure: Original | Simulated | Delta."""
    # Align indices and columns
    all_idx  = sorted(set(pivot_orig.index) | set(pivot_sim.index))
    all_cols = sorted(set(pivot_orig.columns) | set(pivot_sim.columns))

    orig = pivot_orig.reindex(index=all_idx, columns=all_cols, fill_value=0)
    sim  = pivot_sim.reindex(index=all_idx, columns=all_cols, fill_value=0)
    delta = sim - orig

    n_rows = len(all_idx)
    height = max(7, n_rows * 0.42)
    n_cols_display = min(len(all_cols), 20)
    width  = max(24, n_cols_display * 1.2)

    fig = plt.figure(figsize=(width, height), facecolor='#1e1e2e')
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.08)

    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    for ax in axes:
        ax.set_facecolor('#1e1e2e')

    # Restrict columns to top 20 by total for readability
    top_cols = orig.sum().sort_values(ascending=False).head(20).index
    orig_d  = orig[top_cols]
    sim_d   = sim[top_cols]
    delta_d = delta[top_cols]

    kw = dict(fmt='', annot_kws={'size': 8}, linewidths=0.2, linecolor='#333355',
              cbar_kws={'shrink': 0.6})

    # Panel 1 — Original
    sns.heatmap(orig_d, annot=orig_d.map(format_number), cmap='Blues',
                ax=axes[0], **kw)
    axes[0].set_title('Portefeuille Actuel', color='white', fontsize=12, pad=10)

    # Panel 2 — Simulated
    sns.heatmap(sim_d, annot=sim_d.map(format_number), cmap='BuGn',
                ax=axes[1], **kw)
    axes[1].set_title('Portefeuille Simulé', color='#6ef0b0', fontsize=12, pad=10)
    axes[1].set_yticklabels([])
    axes[1].set_ylabel('')

    # Panel 3 — Delta (diverging)
    vmax = max(abs(delta_d.values.max()), abs(delta_d.values.min()), 1)
    sns.heatmap(delta_d, annot=delta_d.map(format_number), cmap='RdYlGn',
                center=0, vmin=-vmax, vmax=vmax, ax=axes[2], **kw)
    axes[2].set_title('Delta (Simulé − Actuel)', color='#f0c06e', fontsize=12, pad=10)
    axes[2].set_yticklabels([])
    axes[2].set_ylabel('')

    for ax in axes:
        ax.tick_params(axis='x', colors='#cccccc', rotation=45, labelsize=7)
        ax.tick_params(axis='y', colors='#cccccc', rotation=0,  labelsize=8)
        ax.xaxis.label.set_color('#cccccc')

    metric_label = {
        'volume': 'Volume (contrats)',
        'concentration_financiere': 'Exposition financière (EUR)',
        'intensite_risk_asset': 'Valeur actifs (EUR)',
    }.get(metric, metric)
    fig.suptitle(
        f'{y_col}  ×  {" | ".join(x_cols)}   —   {metric_label}',
        color='white', fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.show()

print('Helpers loaded.')

Helpers loaded.


In [ ]:
# ── Simulation UI ─────────────────────────────────────────────────────────────

def simulation_ui(df):
    rng = np.random.default_rng(42)

    all_brands    = sorted(df['BRAND_UPDATE'].dropna().unique())
    all_countries = sorted(df['COUNTRY'].dropna().unique())

    # ── Brand table (read-only info) ──────────────────────────────────────
    brand_counts = df['BRAND_UPDATE'].value_counts().reset_index()
    brand_counts.columns = ['Brand', 'Vehicles in portfolio']

    # ── Config widgets ────────────────────────────────────────────────────
    y_w       = widgets.Dropdown(options=['BRAND_UPDATE','POWER_CATEGORY','CO2_BUCKET'],
                                  value='BRAND_UPDATE', description='Axe Y:',
                                  style={'description_width':'70px'})
    x_w       = widgets.SelectMultiple(
                    options=['GROUP_RATING','COUNTERPARTY_RATING','CLS_GROUP_RATING',
                             'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION','SHARED_CLIENT_FLAG'],
                    value=['GROUP_RATING'], rows=5,
                    description='Axe X:', style={'description_width':'70px'})
    m_w       = widgets.Dropdown(
                    options=[('Volume','volume'),
                             ('Concentration Financière','concentration_financiere'),
                             ('Intensité Risk Asset','intensite_risk_asset')],
                    description='Métrique:', style={'description_width':'70px'})
    country_w = widgets.Dropdown(options=['All']+all_countries, value='All',
                                  description='Country:', style={'description_width':'70px'})

    # ── Per-brand add/remove sliders (top 10 brands) ─────────────────────
    top_brands = brand_counts['Brand'].head(10).tolist()
    brand_info_rows = []
    add_sliders    = {}
    remove_sliders = {}

    for brand in top_brands:
        n_current = int(brand_counts.loc[brand_counts['Brand']==brand,'Vehicles in portfolio'].iloc[0])
        lbl = widgets.HTML(f'<b style="width:120px;display:inline-block">{brand}</b> '
                           f'<span style="color:#aaa">({n_current} vehicles)</span>')
        add_s = widgets.IntSlider(min=0, max=max(500, n_current), step=10, value=0,
                                   description='+Add:', continuous_update=False,
                                   layout=widgets.Layout(width='320px'),
                                   style={'description_width':'50px'})
        rem_s = widgets.IntSlider(min=0, max=n_current, step=10, value=0,
                                   description='-Remove:', continuous_update=False,
                                   layout=widgets.Layout(width='320px'),
                                   style={'description_width':'65px'})
        add_sliders[brand]    = add_s
        remove_sliders[brand] = rem_s
        brand_info_rows.append(HBox([lbl, add_s, rem_s]))

    run_btn = widgets.Button(description='Run Simulation',
                              button_style='success',
                              icon='play',
                              layout=widgets.Layout(width='200px', margin='12px 0 0 0'))
    reset_btn = widgets.Button(description='Reset',
                                button_style='warning',
                                icon='refresh',
                                layout=widgets.Layout(width='100px', margin='12px 0 0 4px'))
    out = Output()

    def run_sim(_):
        add_dict    = {b: s.value for b, s in add_sliders.items()    if s.value > 0}
        remove_dict = {b: s.value for b, s in remove_sliders.items() if s.value > 0}

        df_f = df.copy()
        if country_w.value != 'All':
            df_f = df_f[df_f['COUNTRY'] == country_w.value]

        df_sim = apply_simulation(df_f, add_dict, remove_dict, rng)

        pivot_orig = get_grouped_data(df_f,   y_w.value, list(x_w.value), m_w.value)
        pivot_sim  = get_grouped_data(df_sim, y_w.value, list(x_w.value), m_w.value)

        # Summary
        delta_total = len(df_sim) - len(df_f)
        sign = '+' if delta_total >= 0 else ''
        print(f'Portfolio: {len(df_f):,} -> {len(df_sim):,} ({sign}{delta_total})  '
              f'| Added: {sum(add_dict.values())}  Removed: {sum(remove_dict.values())}')

        with out:
            out.clear_output(wait=True)
            plot_three_panels(pivot_orig, pivot_sim, y_w.value, list(x_w.value), m_w.value)

    def reset_sim(_):
        for s in list(add_sliders.values()) + list(remove_sliders.values()):
            s.value = 0
        with out:
            out.clear_output()

    run_btn.on_click(run_sim)
    reset_btn.on_click(reset_sim)

    config_row = HBox([y_w, x_w, m_w, country_w])
    slider_box = widgets.VBox(
        [widgets.HTML('<hr><b>Simulation — Adjust brand volumes:</b>'), *brand_info_rows,
         HBox([run_btn, reset_btn])]
    )

    display(VBox([config_row, slider_box, out]))

    # Show brand table as reference
    display(widgets.HTML('<hr><b>Current portfolio by brand:</b>'))
    display(brand_counts.style.bar(subset=['Vehicles in portfolio'], color='#4a9eff'))


simulation_ui(nova)

HTML(value='<hr><b>Current portfolio by brand:</b>')

,Brand,Vehicles in portfolio
0,PEUGEOT,93594
1,VOLKSWAGEN,83905
2,RENAULT,83394
3,TOYOTA,83391
4,FORD,73358
5,OPEL,72759
6,NISSAN,72749
7,BMW,62753
8,MERCEDES,62497
9,AUDI,52295
